In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from delta import DeltaTable
import os

spark = SparkSession.builder \
    .appName("Delta Lake CDC Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.databricks.delta.optimizeWrite.enabled", "true") \
    .config("spark.databricks.delta.autoOptimize.optimizeIncrementally", "true") \
    .config("spark.ui.showConsoleProgress", "true") \
    .getOrCreate()

print("✓ Spark session created with Delta Lake support")

✓ Spark session created with Delta Lake support


In [3]:
import shutil

# Set up a Delta table path
delta_path = os.path.join("/opt/spark/delta", "delta_cdc_demo")

# Clean up any existing data
if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

df_cdc = spark.createDataFrame(
    [(1, "Alice", "HR"),
     (2, "Bob", "IT"),
     (3, "Carol", "Finance")],
    ["id", "name", "dept"]
)

# Write Delta table with Change Data Feed enabled
(df_cdc.write
 .format("delta")
 .option("delta.enableChangeDataFeed", "true")
 .mode("append")
 .save(delta_path))

print("✓ Delta table created with CDC enabled")

✓ Delta table created with CDC enabled


In [4]:
# Perform updates and deletes
delta_table = DeltaTable.forPath(spark, delta_path)

# Update
delta_table.update(
    condition=col("id") == 2,
    set={"dept": lit("Security")}
)

# Delete
delta_table.delete(condition=col("id") == 3)

print("✓ Update and delete operations complete")

✓ Update and delete operations complete


In [ ]:
# Read change data feed
cdc_df = (spark.read
          .format("delta")
          .option("readChangeFeed", "true")
          .option("startingVersion", 0)
          .load(delta_path))

cdc_df.show(truncate=False)

+---+-----+--------+----------------+---------------+-----------------------+
|id |name |dept    |_change_type    |_commit_version|_commit_timestamp      |
+---+-----+--------+----------------+---------------+-----------------------+
|2  |Bob  |IT      |update_preimage |1              |2026-03-02 17:45:14.746|
|2  |Bob  |Security|update_postimage|1              |2026-03-02 17:45:14.746|
|3  |Carol|Finance |delete          |2              |2026-03-02 17:45:17.182|
|1  |Alice|HR      |insert          |0              |2026-03-02 17:44:26.327|
|2  |Bob  |IT      |insert          |0              |2026-03-02 17:44:26.327|
|3  |Carol|Finance |insert          |0              |2026-03-02 17:44:26.327|
+---+-----+--------+----------------+---------------+-----------------------+



In [8]:
# Write CDC values to CSV
spark_data = "/opt/spark/data"
cdc_output_path = os.path.join(spark_data, "cdc_csv_output")

(cdc_df
 .write
 .mode("overwrite")
 .option("header", "true")
 .csv(cdc_output_path))

print(f"✓ CDC values written to CSV at: {cdc_output_path}")

✓ CDC values written to CSV at: /opt/spark/data/cdc_csv_output


In [9]:
cdc_df.show(truncate=False)

+---+-----+--------+----------------+---------------+-----------------------+
|id |name |dept    |_change_type    |_commit_version|_commit_timestamp      |
+---+-----+--------+----------------+---------------+-----------------------+
|2  |Bob  |IT      |update_preimage |1              |2026-03-02 17:45:14.746|
|2  |Bob  |Security|update_postimage|1              |2026-03-02 17:45:14.746|
|3  |Carol|Finance |delete          |2              |2026-03-02 17:45:17.182|
|1  |Alice|HR      |insert          |0              |2026-03-02 17:44:26.327|
|2  |Bob  |IT      |insert          |0              |2026-03-02 17:44:26.327|
|3  |Carol|Finance |insert          |0              |2026-03-02 17:44:26.327|
+---+-----+--------+----------------+---------------+-----------------------+



In [10]:
cdc_df.filter(col("_change_type") == "update_postimage").show(truncate=False)

+---+----+--------+----------------+---------------+-----------------------+
|id |name|dept    |_change_type    |_commit_version|_commit_timestamp      |
+---+----+--------+----------------+---------------+-----------------------+
|2  |Bob |Security|update_postimage|1              |2026-03-02 17:45:14.746|
+---+----+--------+----------------+---------------+-----------------------+



In [ ]:
cdc_df = (spark.read
          .format("delta")
          .option("readChangeFeed", "true")
          .option("startingVersion", 0)
          .load(delta_path))
cdc_df.filter(col("_change_type").isin("update_postimage", "insert")).show(truncate=False)